<a href="https://colab.research.google.com/github/Amruth-U-tech/DL-Journey/blob/main/03-CNN/VGG16-TransferLearning-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**VGG16 with transfer learning**
we can recreate the vgg-16 model but y do it when its already available

note that recreating vgg16 cannot give u the same accuracy as promised because it was trained on huge data set with 1000+ classes, hence any smaller data set cannot give the same accuracy u can increase the accuracy by


1.   batch normalization and drop out
2.   apllying GAP grobal aveergae pooling instead of Flatten
3.   use transfer learning



In [2]:
#we first import the packages required to do data preprocessing
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings

warnings.filterwarnings("ignore")  #this is to supress any warnings we get

train_dir = "/content/drive/MyDrive/alzheimer dataset/train"
test_dir = "/content/drive/MyDrive/alzheimer dataset/test"

In [3]:
#now we craerte an image generator object which we take the image path and get through the agumentation
#and preprocessing

train_datagen = ImageDataGenerator(rescale=1./255,
                                   shear_range=0.4,
                                   zoom_range=0.2,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   rotation_range=90,
                                   horizontal_flip=True,
                                   fill_mode="nearest"
                                   )
test_datagen = ImageDataGenerator(rescale=1./255)

#loading image and lables, ensuring grayscale

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128,128),            #resizing the image to 128x128
    color_mode="grayscale",
    class_mode="categorical"
)

test_generator = test_datagen.flow_from_directory(test_dir,target_size=(128,128),color_mode="grayscale",class_mode="categorical")

Found 4648 images belonging to 4 classes.
Found 1232 images belonging to 4 classes.


**setting up VGG16 to our model**

when we import the VGG16 from the lib we get the original architecture as it is including the input image size,conv layers, and flattening method everything

but we have our own requirements thats is imput img size is 128x128 with single channel etc

so here we are making some modifications into VGG16 and our imgs together do that we get effcient traing

1.  we are converting our imgs to 3 channels
2. we are removing the top part og VGG16 cause those original layer are prepared accordingly for rgb with maybe other dimensions not 128x128 so we remove the top layer that is the dense layer or fully connected NN (by removing it we can define our own classifier with our own density of dense layers cause we just need 4 neurons in output not 1000+)

In [8]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Input, Concatenate
from tensorflow.keras.layers import Dense,Flatten,Dropout
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam

#1. input for grayscale images
input_tensor = Input(shape=(128,128,1))

#2. converting grayscale --> rgb we are actually adding more channels
input_tensor_rgb = Concatenate()([input_tensor,input_tensor,input_tensor]) # we made it into 3 channels imgs

#3. loading VGG16 without the top layers
vgg16 = VGG16(weights="imagenet",  #this tells that we want weights of orginal VGG16 that wa strained on imagenet to be retained
              include_top=False,   #removing the dense layer (fully connected neural network)
              input_tensor = input_tensor_rgb) #giving and input as we give in other layer formation

#now finally we have VGG16 where we can have our own inpiut size, output classifier with the weights that is the original kernels retained

#4. now we build the the model first with the convolution layer that is retained is loaded
modelvgg16 = Sequential()
modelvgg16.add(vgg16) #cause vgg16 is the instance which holds customized version of VGG16 so we add that to our model

#5. as there is no fully connected NN we make it according to our req
modelvgg16.add(Flatten())
modelvgg16.add(Dense(256,activation="relu"))
modelvgg16.add(Dropout(0.5))
modelvgg16.add(Dense(128,activation="relu"))
modelvgg16.add(Dropout(0.5))

#final output
modelvgg16.add(Dense(4,activation="softmax"))

#HERE COMES THE MOST IMPORTANT PART
'''Dont u think that the orginal kernels will get trained to our dataset and the kernels values
might get disturbed during the fine tuning to ur dataset hence we freeze the orginal kernels (layers)
so that they are not trainable at all hence cannot be changed, if ther smth to be learned it will be
from the fully connected NN where the weights must get updated accordingly'''

#6. freeze the layers of vgg16 instance that is already carrying the original VGG16
for layer in vgg16.layers:
  layer.trainable = False

#7. now we complie the model
modelvgg16.compile(
    optimizer = Adam(learning_rate=0.00001),
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

modelvgg16.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,845,508 (64.26 MB)

 Trainable params: 2,130,820 (8.13 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

**setup complete**

the VGG16 setup is complete and now u can fine tune to train on ur dataset
and u will see a drastic improvement than just recreating it

to furthur improve the training accuracy u can do the following

1.  adding additional 2 convolutional layers after all the vgg16 conv layers

BUT NOTE: if u are adding ur own layers the using the freeze by for loop will freeze all the kernals including ur own added kernels

so u freeze the vgg16 kernels before hand here:
 vgg16=VGG16()
 vgg16.trainable = False

 then later u dont have to freeze anything else and ur convolutional layer is trainable

 2. aplly batch normalisation and play around with what u have added nothing can be changed in vgg16
 3. u can add gobal avg pooling also

In [8]:
#modelvgg16.fit(train_generator <where the img is agumented>,epochs=50)
#loss,vgg16acc = modelvgg16.evaluate(test_generator)